## Structured output

Models can be requested to provide their response in a format matching a given schema . This is useful for ensuring the output can be easily parsed and used in subsequest processing . Langchain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation , descriptions and nested structures.

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model('groq:qwen/qwen3-32b')
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A27517E120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A27517EE40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel , Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="the director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A27517E120>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A27517EE40>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': "The movie's rating ou

In [6]:
model.invoke("Provide details about the movie Inception")   ## not structured output , the output is as the LLM wishes 

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It\'s a 2010 film directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title is Inception, which I think has something to do with entering someone\'s mind. \n\nThe plot probably involves dreams and reality, maybe some sort of heist. I remember the director is known for complex narratives and time manipulation, like in Memento and Interstellar. Inception might have a similar structure, with layers of reality or something. \n\nThe main character, played by DiCaprio, might be a criminal who enters people\'s dreams to steal information. The term "inception" could mean planting an idea in someone\'s subconscious. So maybe the story is about a team trying to plant an idea instead of stealing. \n\nThe cast includes other big names. Tom Hardy, maybe as a member of the team. Ellen Page was in a Nolan film before, in Batman Begins, so

In [7]:
response = model_with_structure.invoke("Provide details about the movie inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [8]:
from pydantic import BaseModel , Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="the director of the movie")
    rating:float=Field(...,description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie , include_raw=True) ## Include_raw displays the ai message 
response = model_with_structure.invoke("Provide details about the movie Intersteller.")
response    

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Interstellar." Let me see what I need to do here. They provided a function called Movie with parameters: director, rating, title, and year. All of these are required.\n\nFirst, I need to recall the information about "Interstellar." The title is obviously "Interstellar." It was released in 2014, so the year is 2014. The director is Christopher Nolan. As for the rating, I think it\'s around 8.6 on IMDb, but since the function expects a number out of 10, maybe I should use that. Wait, but the user didn\'t specify the source of the rating. Hmm, but the function just says "the movie\'s rating out of 10," so I can use the IMDb rating. Let me confirm: yes, IMDb has it at 8.6. \n\nSo putting it all together, the arguments should be title: "Interstellar", year: 2014, director: "Christopher Nolan", and rating: 8.6. I need to make sure all required fields are included. The 

# Nested Structure

In [9]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor] = []
    genres: list[str] = []
    budget: float | None = Field(None, description="budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie Inception.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Action', 'Science Fiction', 'Heist'], budget=160.0)

# TypedDict

TypedDict provides a simpler alternative using Python's built-in typing , ideal when you don't need runtime validation.

In [10]:
from typing_extensions import TypedDict , Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ...,"The title of the movie"]
    year: Annotated[int,...,"The year the movie was released"]
    directed: Annotated[str,...,"The director of the movie"]
    rating: Annotated[float,...,"the movie's rating out of 10"]

In [16]:
model_with_typedict = model.with_structured_output(MovieDict)
response = model_with_typedict.invoke("provide details about the movie Harry potter ")

In [17]:
response

{'directed': 'Chris Columbus',
 'rating': 7.6,
 'title': "Harry Potter and the Sorcerer's Stone",
 'year': 2001}

In [18]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor] = []
    genres: list[str] = []
    budget: float | None = Field(None, description="budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about the movie Inception.")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'}],
 'genres': ['Science Fiction', 'Action'],
 'title': 'Inception',
 'year': 2010}

In [19]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

# DataClasses

A data class is a class typically containing mainly data , although there arent really any restrictions . you create it using the @dataclass decorator

In [20]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

In [21]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from typing_extensions import TypedDict


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [23]:
## dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information of a person."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = model,
    response_format = ContactInfo
)    

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})


result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')